In [3]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate


In [4]:
model = OllamaLLM(model="llama3.1:8b")

In [5]:
template = """
You are an expert in answering questions about pizzas

Here are some relevant reviews: {reviews}

Here is the quesstion to answer: {questions}
"""

prompt = ChatPromptTemplate.from_template(template)
chain = prompt | model

##test
chain.invoke({"reviews":["Lafamilia is the best pizza place in town"],"questions":"What is the best pizza place in town?"})

"That's an easy one! According to the review, Lafamilia is the best pizza place in town. I'd recommend checking it out if you're looking for a great pizza experience!"

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
### other model test

model_2 = OllamaLLM(model=os.getenv(MODEL_2))

NameError: name 'MODEL_2' is not defined

In [ ]:
## Write in a python script
'''
while True:
    question = input("Ask your question (q to quit):")
    if question == "q":
        break
    
    result = chain.invoke({"reviews":["Lafamilia is the best pizza place in town"],"questions":question})
    
'''

In [15]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
import os
import pandas as pd

In [16]:
dir = "data/csv/RRr.csv"
df = pd.read_csv(dir)

In [17]:
embeddings = OllamaEmbeddings(model="mxbai-embed-large:v1")

In [19]:
db_location = "./chrome_langchain_db"
add_documents = not os.path.exists(db_location)

if add_documents:
    documents = []
    ids = []
    
    for i, row in df.iterrows():
        document = Document(
            page_content = row["Title"] + " " + row["Review"],
            metadata = {"rating": row["Rating"], "date":row["Date"]},
            id= str(i)
        )
        ids.append(str(i))
        documents.append(document)
    
    
vector_store = Chroma(
    collection_name = "restaurant_reviews",
    persist_directory=db_location,
    embedding_function = embeddings
)
        
if add_documents:
    vector_store.add_documents(documents=documents,ids = ids)


In [20]:
retriever = vector_store.as_retriever(
    search_kwargs = {"k":5}
)

In [21]:
question = "Name some good pizza shops"

reviews = retriever.invoke(question)

chain.invoke({"reviews":reviews,"questions":question})

'Based on these reviews, it seems like there are at least a few good pizza shops around!\n\nFrom what I\'ve gathered, the top-rated pizzeria based on these reviews is... (dramatic pause) ...the one that has been consistently praised for its "supreme quality control" and serving up pizzas with perfect crusts and delicious toppings! They seem to have a strong reputation for consistency and high-quality food. Review #112 specifically mentions their impressive ability to deliver identical, excellent pizzas every time.\n\nAdditionally, Review #0 praises the shop\'s signature pepperoni pizza as having the "perfect ratio of sauce to cheese" and "the perfect crust on the outside and chewy inside." So, it seems like this place knows how to make a great pie!\n\nReview #28 also mentions that their menu has "something for everyone," which might indicate that they have a diverse range of options available.\n\nAs for specific shop names... unfortunately, none are mentioned in these reviews. But base